# IMPORT AND PREPARE DATA

In [ ]:
# Import initial dependencies
import pandas as pd
import numpy as np

In [ ]:
# Convert CSVs to DFs
features = "../output/UPDATED_reduced_wine_features.csv"
targets = "../output/reduced_wine_targets.csv"

features_df_pre = pd.read_csv(features)
targets_df =pd.read_csv(targets) 

In [ ]:
features_df = features_df_pre[['white', 'flavor_categories_taste_notes_light, fruity', 'price', 'country_Austria']]

In [ ]:
# Check feature DF head
features_df.head()

In [ ]:
# Check feature DF shape
features_df.shape

In [ ]:
# Check targets DF
targets_df.head()

In [ ]:
# Check Dtypes
targets_df.dtypes

In [ ]:
# Check feature and target DF shape
print(features_df.shape, targets_df.shape)

In [ ]:
# Assign the features and target to X and y abd check assignments
raw_feature_data = features_df.values
raw_target_data = targets_df.values
X = raw_feature_data[:, 0:5]
y = raw_target_data.reshape(-1,1)

print(X, y)

# SPLIT, NORMALIZE, AND ENCODE THE DATA

In [ ]:
# Create the train and test sets for the features and target
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

In [ ]:
# Scale the features using the MinMax scaler since we know their values
from sklearn.preprocessing import LabelEncoder, MinMaxScaler

X_scaler = MinMaxScaler().fit(X_train)
X_train_scaled = X_scaler.transform(X_train)
X_test_scaled = X_scaler.transform(X_test)

y_scaler = MinMaxScaler().fit(y_train)
y_train_scaled = y_scaler.transform(y_train)
y_test_scaled = y_scaler.transform(y_test)

# HYPERPERAMETER TUNING

In [ ]:
# Import the sequential and dense modules fo build my NN
from sklearn.model_selection import GridSearchCV
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import Dropout
from keras.wrappers.scikit_learn import KerasRegressor
from keras.constraints import maxnorm

# Run XGB Regressor with scalled data and recommended parameters from the previous test

In [ ]:
print('Select Model...')
from xgboost import XGBRegressor
from datetime import datetime
import pprint
import os

# Create start time and model for XGB Regressor
start_time  = datetime.now()
xgb_reg = XGBRegressor() 


parameters_reg = {'n_estimators': [850], 'max_depth':[5, 7], \
              'learning_rate':[.01], 'batch_size':[10, 16]}

grid_search = GridSearchCV(estimator=xgb_reg, param_grid=parameters_reg, cv=10, n_jobs=-1)


print("parameters:")
pprint.pprint(parameters_reg)


grid = grid_search.fit(X_train_scaled, y_train_scaled.ravel())


print("Best score: %0.3f" % grid.best_score_)
print("Best parameters set:")
best_parameters=grid.best_estimator_.get_params()


for param_name in sorted(best_parameters.keys()):
    print("\t%s: %r" % (param_name, best_parameters[param_name]))
    end_time = datetime.now()
print(f'Select Done..., Time Cost: {((end_time - start_time).seconds)}.') 

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score

# Use our model to predict a value
predicted = grid.predict(X_test_scaled)

# Score the prediction with mse and r2
mse = mean_squared_error(y_test_scaled, predicted)
r2 = r2_score(y_test_scaled, predicted)

# Print scores
print(f"Mean Squared Error (MSE): {mse}")
print(f"R-squared (R2): {r2}")

In [ ]:
# Create list of predictions and check
predicted_list = grid.predict(X_test_scaled)

predicted_list

In [ ]:
# Create list of actual values and check
actual_list = y_test_scaled

actual_list

# PRINT PREDICTIONS FOR SCALED DATA TO COMPARE TO ACTUAL RATINGS

In [ ]:
# Create DF for unscaled test to further compare predictions with actual points
prediction_comparrisons_df = pd.DataFrame({'XGB_Predictions (unscaled)' : predicted_list,
                                           'Actual_Points (unscaled)' : actual_list.ravel()
                                          })

prediction_comparrisons_df.head(40)

In [ ]:
# Export unscaled results
prediction_comparrisons_df.to_csv(r'C:\Users\howar\Desktop\Data Science Boot Camp\Group_Project_3\prediction_modeling\output\UPDATED_predictions_vs_actual.csv', index=False)

In [ ]:
# Save the tuned model for future use
import joblib
filename = 'saved_models\XGB_Regressor_unscaled_(BEST_SCALED_REDUCED_FEATURES).sav'
joblib.dump(grid, filename)

# Run XGB Regressor with unscalled data and recommended parameters from the previous test

In [ ]:
print('Select Model...')
from xgboost import XGBRegressor
from datetime import datetime
import pprint
import os


xgb_reg_2 = XGBRegressor() 


parameters_reg_2 = {'n_estimators': [850], 'max_depth':[5], \
              'learning_rate':[.01], 'batch_size':[10]}

grid_search_2 = GridSearchCV(estimator=xgb_reg_2, param_grid=parameters_reg_2, cv=10, n_jobs=-1)


print("parameters:")
pprint.pprint(parameters_reg_2)


grid_2 = grid_search_2.fit(X_train, y_train.ravel())


print("Best score: %0.3f" % grid_2.best_score_)
print("Best parameters set:")
best_parameters_2=grid_2.best_estimator_.get_params()


for param_name in sorted(parameters_reg_2.keys()):
    print("\t%s: %r" % (param_name, best_parameters_2[param_name]))



In [ ]:
from sklearn.metrics import mean_squared_error, r2_score

# Use our model to predict a value
predicted = grid_2.predict(X_test)

# Score the prediction with mse and r2
mse = mean_squared_error(y_test, predicted)
r2 = r2_score(y_test, predicted)

# Print scores
print(f"Mean Squared Error (MSE): {mse}")
print(f"R-squared (R2): {r2}")

In [ ]:
# Create list of predictions and check
predicted_list_unscaled = grid_2.predict(X_test)

predicted_list_unscaled

In [ ]:
# Create list of actual values and check
actual_list_unscaled = y_test

actual_list_unscaled

# PRINT PREDICTIONS AND ACTUAL RATINGS FOR UNSCALED DATA

In [ ]:
# Create DF for unscaled test to further compare predictions with actual points
prediction_comparrisons_unscaled_df = pd.DataFrame({'XGB_Predictions (unscaled)' : predicted_list_unscaled,
                                           'Actual_Points (unscaled)' : actual_list_unscaled.ravel()
                                          })

prediction_comparrisons_unscaled_df.head(40)

In [ ]:
# Export unscaled results
prediction_comparrisons_unscaled_df.to_csv(r'C:\Users\howar\Desktop\Data Science Boot Camp\Group_Project_3\prediction_modeling\output\UPDATED_predictions_vs_actual_unscaled.csv', index=False)

In [ ]:
# Save the tuned model for future use
import joblib
filename = 'saved_models\XGB_Regressor_unscaled_(BEST_UNSCALED_REDUCED_FEATURES).sav'
joblib.dump(grid_2, filename)

In [ ]:
# Export model to a serialized Python object structure to use in the App.py file
import pickle
pickle.dump(grid_2, open("../output/XGB_unscaled_model.pkl","wb"))

In [ ]:
# Export model to a serialized Python object structure to use in the App.py file (this time, in the app.py directory...)
import pickle
pickle.dump(grid_2, open("../../wine_application/parallax-template/XGB_unscaled_model.pkl","wb"))

In [ ]:
# Create list of predictions and check
predicted_list_full = grid_2.predict(X)

predicted_list_full

In [ ]:
predicted_list_full.shape

In [ ]:
# Create list of actual values and check
actual_list_full = y

actual_list_full

In [ ]:
actual_list_full.shape

# EXPORT FINAL PREDICTED FEATURES AND TARGETS IN ONE COMBINED FILE TO USE IN THE APPLICATION

In [ ]:
features_df['predicted_scores'] = predicted_list_full

In [ ]:
features_df.columns

In [ ]:
features_df.head()

In [ ]:
features_df.shape

In [ ]:
# Export unscaled results
features_df.to_csv(r'C:\Users\howar\Desktop\Data Science Boot Camp\Group_Project_3\prediction_modeling\output\full_predictions.csv', index=False)